In [0]:
%pip install xgboost

In [0]:
import pandas as pd
import numpy as np
import pickle
import boto3
import json
import time
from io import BytesIO
from datetime import datetime, timezone

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score
from xgboost import XGBRegressor

# Control de promoción de modelo
AUTO_PROMOTE_IF_BETTER = True
FORCE_PROMOTE_TO_CHAMPION = False

# =============================================================
# 1. CARGAR DATOS — Silver deduped (preferido) ó Gold legacy
# =============================================================
try:
    config = {
        "aws_access_key": dbutils.secrets.get(scope="aws", key="access_key"),
        "aws_secret_key": dbutils.secrets.get(scope="aws", key="secret_key"),
        "bucket_name": "bronce-scrap-date",
    }
except Exception:
    with open("aws_secrets.json", "r") as f:
        config = json.load(f)

BUCKET = config["bucket_name"]
S3_OPTS = {
    "fs.s3a.access.key": config["aws_access_key"],
    "fs.s3a.secret.key": config["aws_secret_key"],
    "fs.s3a.endpoint": "s3.amazonaws.com",
}

try:
    ruta_deduped = f"s3a://{BUCKET}/silver/master_deduped/"
    reader = spark.read.format("delta")
    for k, v in S3_OPTS.items():
        reader = reader.option(k, v)
    df_spark = reader.load(ruta_deduped)
    print(f"📊 Leyendo Silver Deduped (Delta): {ruta_deduped}")
except Exception:
    ruta_gold = f"s3a://{BUCKET}/gold/app_inmuebles/"
    reader = spark.read.format("parquet")
    for k, v in S3_OPTS.items():
        reader = reader.option(k, v)
    df_spark = reader.load(ruta_gold)
    print(f"📊 Fallback → Gold legacy (Parquet): {ruta_gold}")

print(f"   Columnas: {df_spark.columns}")

# =============================================================
# 2. PREPARAR FEATURES BASE
# =============================================================
available = set(df_spark.columns)

base_numeric_candidates = [
    "area_m2", "habitaciones", "banos", "garajes",
    "num_portales", "dispersion_pct_grupo", "precio_desviacion_grupo_pct",
    "data_completeness",
]
cols_numeric_base = [c for c in base_numeric_candidates if c in available]
cols_categorical = [
    c for c in ["tipo_inmueble", "estado_inmueble", "fuente", "city_token"]
    if c in available
]
cols_text = [c for c in ["ubicacion_norm", "titulo"] if c in available]
if "ubicacion_norm" not in available and "ubicacion_raw" in available:
    cols_text = [c for c in ["ubicacion_raw", "titulo"] if c in available]

all_cols = ["precio_num"] + cols_numeric_base + cols_categorical + cols_text
print(f"📋 Features base: numeric={cols_numeric_base}, cat={cols_categorical}, text={cols_text}")

df = df_spark.select(*all_cols).toPandas()
print(f"   Total registros: {len(df):,}")

for col in cols_numeric_base + ["precio_num"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

for col in cols_categorical:
    df[col] = df[col].fillna("desconocido").astype(str)

if "city_token" not in df.columns:
    df["city_token"] = "otra_ciudad"
if "tipo_inmueble" not in df.columns:
    df["tipo_inmueble"] = "otro"
if "fuente" not in df.columns:
    df["fuente"] = "desconocido"

conteos = df["city_token"].value_counts()
ciudades_validas = conteos[conteos >= 80].index
df["city_token"] = df["city_token"].where(df["city_token"].isin(ciudades_validas), other="otra_ciudad")
print(f"   city_token: {df['city_token'].nunique()} categorías después de colapso (umbral=80)")

tipos_validos = df["tipo_inmueble"].value_counts()
tipos_validos = tipos_validos[tipos_validos >= 40].index
df["tipo_inmueble"] = df["tipo_inmueble"].where(df["tipo_inmueble"].isin(tipos_validos), other="otro")

text_parts = [df[col].fillna("") for col in cols_text]
if text_parts:
    df["texto_completo"] = text_parts[0]
    for part in text_parts[1:]:
        df["texto_completo"] = df["texto_completo"] + " " + part
else:
    df["texto_completo"] = ""

# =============================================================
# 3. FILTRAR OUTLIERS — precio_m2 por segmento mercado
# =============================================================
df = df[(df["area_m2"] >= 20) & (df["area_m2"] <= 800) & (df["precio_num"] > 0)].copy()
df["precio_m2"] = df["precio_num"] / df["area_m2"]
df["market_segment"] = df["city_token"].astype(str) + "__" + df["tipo_inmueble"].astype(str)
df["log_area_m2"] = np.log1p(df["area_m2"])
df["banos_por_habitacion"] = (df["banos"] / df["habitaciones"].replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)

segment_bounds = (
    df.groupby("market_segment")
    .agg(
        segment_n=("precio_m2", "size"),
        pm2_p10=("precio_m2", lambda s: s.quantile(0.10)),
        pm2_p90=("precio_m2", lambda s: s.quantile(0.90)),
    )
    .reset_index()
)
segment_bounds["pm2_iqr"] = segment_bounds["pm2_p90"] - segment_bounds["pm2_p10"]
segment_bounds["pm2_lower"] = (segment_bounds["pm2_p10"] - 1.5 * segment_bounds["pm2_iqr"]).clip(lower=0)
segment_bounds["pm2_upper"] = segment_bounds["pm2_p90"] + 1.5 * segment_bounds["pm2_iqr"]

df_filtered = df.merge(
    segment_bounds[["market_segment", "segment_n", "pm2_lower", "pm2_upper"]],
    on="market_segment",
    how="left"
)
df_filtered = df_filtered[
    (df_filtered["segment_n"] < 25) |
    (df_filtered["precio_m2"].between(df_filtered["pm2_lower"], df_filtered["pm2_upper"]))
]
global_p02 = df_filtered["precio_m2"].quantile(0.02)
global_p98 = df_filtered["precio_m2"].quantile(0.98)
df_clean = df_filtered[df_filtered["precio_m2"].between(global_p02, global_p98)].copy()
df_clean = df_clean.drop(columns=["segment_n", "pm2_lower", "pm2_upper"], errors="ignore")
print(f"🧹 Outlier filter (precio_m2 por segmento): {len(df_clean):,} registros (de {len(df):,})")

print("\n" + "=" * 60)
print("  DIAGNÓSTICO DE DISTRIBUCIÓN DE PRECIOS")
print("=" * 60)
print(df_clean["precio_num"].describe(percentiles=[.05, .10, .25, .50, .75, .90, .95, .99]).to_string())
print(f"\n  Mediana:  ${df_clean['precio_num'].median() / 1e6:.0f}M")
print(f"  Media:    ${df_clean['precio_num'].mean() / 1e6:.0f}M")
print(f"  Skewness: {df_clean['precio_num'].skew():.2f}")
print(f"  precio_m2 mediano: ${df_clean['precio_m2'].median() / 1e6:.2f}M")

# =============================================================
# 4. PREPARACIÓN DE FEATURES LEAK-SAFE
# =============================================================
cols_numeric_base = [
    c for c in [
        "area_m2", "log_area_m2", "habitaciones", "banos", "garajes",
        "banos_por_habitacion", "num_portales", "dispersion_pct_grupo",
        "precio_desviacion_grupo_pct", "data_completeness",
    ]
    if c in df_clean.columns and df_clean[c].notna().mean() > 0.20
]
cols_categorical = [
    c for c in ["tipo_inmueble", "estado_inmueble", "fuente", "city_token"]
    if c in df_clean.columns and df_clean[c].nunique() > 1
]

feature_cols_base = cols_numeric_base + cols_categorical + ["texto_completo", "market_segment"]
X_raw = df_clean[feature_cols_base].copy()
y = df_clean["precio_num"].copy()

sample_weight = pd.Series(1.0, index=df_clean.index)
if "num_portales" in df_clean.columns:
    sample_weight += df_clean["num_portales"].fillna(1).clip(lower=1, upper=4).sub(1) * 0.20
if "dispersion_pct_grupo" in df_clean.columns:
    sample_weight *= np.where(df_clean["dispersion_pct_grupo"].fillna(0) <= 15, 1.10, 0.95)
    sample_weight *= np.where(df_clean["dispersion_pct_grupo"].fillna(0) >= 35, 0.85, 1.00)
sample_weight = sample_weight.clip(lower=0.70, upper=1.60)

X_train_raw, X_test_raw, y_train, y_test, w_train, w_test = train_test_split(
    X_raw, y, sample_weight, test_size=0.2, random_state=42
)
print(f"\nTrain: {len(X_train_raw):,} | Test: {len(X_test_raw):,}")

def compute_market_features(train_x, train_y, apply_x):
    train_market = train_x[["city_token", "market_segment", "fuente", "area_m2", "habitaciones"]].copy()
    train_market["precio_num"] = train_y.values
    train_market = train_market.dropna(subset=["city_token", "market_segment", "fuente", "area_m2", "precio_num"])
    train_market = train_market[train_market["area_m2"] > 0].copy()
    train_market["precio_m2_train"] = train_market["precio_num"] / train_market["area_m2"]
    train_market["habitaciones_bucket"] = train_market["habitaciones"].fillna(-1).clip(-1, 6)

    pm2_p01 = train_market["precio_m2_train"].quantile(0.01)
    pm2_p99 = train_market["precio_m2_train"].quantile(0.99)
    train_market["precio_m2_train"] = train_market["precio_m2_train"].clip(pm2_p01, pm2_p99)

    city_stats = (
        train_market
        .groupby("city_token")
        .agg(
            precio_mediano_ciudad=("precio_num", "median"),
            precio_m2_mediano_ciudad=("precio_m2_train", "median"),
            area_mediana_ciudad=("area_m2", "median"),
        )
        .reset_index()
    )

    segment_stats = (
        train_market
        .groupby("market_segment")
        .agg(
            precio_m2_mediano_segmento=("precio_m2_train", "median"),
            precio_mediano_segmento=("precio_num", "median"),
        )
        .reset_index()
    )

    hab_stats = (
        train_market
        .groupby(["city_token", "habitaciones_bucket"])
        .agg(precio_m2_mediano_habs=("precio_m2_train", "median"))
        .reset_index()
    )

    source_stats = train_market.merge(segment_stats, on="market_segment", how="left")
    source_stats["ratio_vs_segment"] = source_stats["precio_num"] / (
        source_stats["area_m2"] * source_stats["precio_m2_mediano_segmento"].replace(0, np.nan)
    )
    source_stats["ratio_vs_segment"] = source_stats["ratio_vs_segment"].replace([np.inf, -np.inf], np.nan)
    source_stats = source_stats.dropna(subset=["ratio_vs_segment"])
    source_stats["ratio_vs_segment"] = source_stats["ratio_vs_segment"].clip(
        source_stats["ratio_vs_segment"].quantile(0.01),
        source_stats["ratio_vs_segment"].quantile(0.99),
    )

    fuente_ratio_stats = (
        source_stats.groupby("fuente")
        .agg(
            fuente_factor=("ratio_vs_segment", "median"),
            fuente_n=("ratio_vs_segment", "size"),
        )
        .reset_index()
    )

    fuente_segmento_ratio_stats = (
        source_stats.groupby(["fuente", "market_segment"])
        .agg(
            fuente_segmento_factor=("ratio_vs_segment", "median"),
            fuente_segmento_n=("ratio_vs_segment", "size"),
        )
        .reset_index()
    )

    result = apply_x.copy()
    result["habitaciones_bucket"] = result["habitaciones"].fillna(-1).clip(-1, 6)
    result = result.merge(city_stats, on="city_token", how="left")
    result = result.merge(segment_stats, on="market_segment", how="left")
    result = result.merge(hab_stats, on=["city_token", "habitaciones_bucket"], how="left")
    result = result.merge(fuente_ratio_stats, on="fuente", how="left")
    result = result.merge(fuente_segmento_ratio_stats, on=["fuente", "market_segment"], how="left")

    global_price_median = train_market["precio_num"].median()
    global_pm2_median = train_market["precio_m2_train"].median()
    global_area_median = train_market["area_m2"].median()
    global_fuente_factor = float(source_stats["ratio_vs_segment"].median()) if len(source_stats) else 1.0

    result["precio_mediano_ciudad"] = result["precio_mediano_ciudad"].fillna(global_price_median)
    result["precio_m2_mediano_ciudad"] = result["precio_m2_mediano_ciudad"].fillna(global_pm2_median)
    result["area_mediana_ciudad"] = result["area_mediana_ciudad"].fillna(global_area_median)
    result["precio_m2_mediano_segmento"] = result["precio_m2_mediano_segmento"].fillna(result["precio_m2_mediano_ciudad"])
    result["precio_mediano_segmento"] = result["precio_mediano_segmento"].fillna(result["precio_mediano_ciudad"])
    result["precio_m2_mediano_habs"] = result["precio_m2_mediano_habs"].fillna(result["precio_m2_mediano_segmento"])
    result["fuente_factor"] = result["fuente_factor"].fillna(global_fuente_factor)
    result["fuente_segmento_factor"] = result["fuente_segmento_factor"].fillna(result["fuente_factor"])
    result["fuente_segmento_factor"] = np.where(
        result["fuente_segmento_n"].fillna(0) >= 25,
        result["fuente_segmento_factor"],
        result["fuente_factor"]
    )

    result["precio_estimado_ciudad_area"] = result["area_m2"] * result["precio_m2_mediano_ciudad"]
    result["precio_estimado_segmento_area"] = result["area_m2"] * result["precio_m2_mediano_segmento"]
    result["precio_estimado_segmento_area_ajustado"] = result["precio_estimado_segmento_area"] * result["fuente_segmento_factor"]
    result["precio_estimado_segmento_area_ajustado"] = result["precio_estimado_segmento_area_ajustado"].fillna(result["precio_estimado_segmento_area"])
    result["precio_estimado_segmento_area_ajustado"] = result["precio_estimado_segmento_area_ajustado"].fillna(result["precio_estimado_ciudad_area"])
    result["area_vs_ciudad_ratio"] = result["area_m2"] / result["area_mediana_ciudad"].replace(0, np.nan)
    result["area_vs_ciudad_ratio"] = result["area_vs_ciudad_ratio"].replace([np.inf, -np.inf], np.nan)
    result["ajuste_fuente_pct"] = (result["fuente_segmento_factor"] - 1.0) * 100.0

    return result, city_stats, segment_stats, fuente_ratio_stats, fuente_segmento_ratio_stats, {
        "global_price_median": global_price_median,
        "global_pm2_median": global_pm2_median,
        "global_area_median": global_area_median,
        "global_fuente_factor": global_fuente_factor,
        "pm2_clip_p01": pm2_p01,
        "pm2_clip_p99": pm2_p99,
    }

print("\n🏙️ Calculando features de mercado (solo desde train)...")
X_train_enriched, city_stats, segment_stats, fuente_ratio_stats, fuente_segmento_ratio_stats, market_meta = compute_market_features(X_train_raw, y_train, X_train_raw)
X_test_enriched, _, _, _, _, _ = compute_market_features(X_train_raw, y_train, X_test_raw)

market_features = [
    "precio_mediano_ciudad",
    "precio_m2_mediano_ciudad",
    "precio_m2_mediano_segmento",
    "precio_m2_mediano_habs",
    "precio_estimado_ciudad_area",
    "precio_estimado_segmento_area",
    "precio_estimado_segmento_area_ajustado",
    "fuente_factor",
    "fuente_segmento_factor",
    "ajuste_fuente_pct",
    "area_vs_ciudad_ratio",
]
cols_numeric_final = cols_numeric_base + market_features

for col in market_features:
    train_nan = int(X_train_enriched[col].isna().sum())
    test_nan = int(X_test_enriched[col].isna().sum())
    print(f"  {col:34s} train NaN={train_nan} | test NaN={test_nan}")

X_train_enriched["area_vs_ciudad_ratio"] = X_train_enriched["area_vs_ciudad_ratio"].fillna(1.0)
X_test_enriched["area_vs_ciudad_ratio"] = X_test_enriched["area_vs_ciudad_ratio"].fillna(1.0)

unseen_test_cities = int((~X_test_raw["city_token"].isin(X_train_raw["city_token"].unique())).sum())
print(f"  Ciudades no vistas en test: {unseen_test_cities}")

X_train = X_train_enriched[cols_numeric_final + cols_categorical + ["texto_completo"]].copy()
X_test = X_test_enriched[cols_numeric_final + cols_categorical + ["texto_completo"]].copy()

def build_model():
    transformers = [
        (
            "txt",
            TfidfVectorizer(
                max_features=100,
                ngram_range=(1, 2),
                min_df=3,
                stop_words=[
                    "en", "de", "la", "el", "y", "con", "se", "del",
                    "por", "los", "las", "un", "una", "para", "al",
                ],
                sublinear_tf=True,
            ),
            "texto_completo",
        ),
        ("num", SimpleImputer(strategy="median"), cols_numeric_final),
    ]

    if cols_categorical:
        transformers.append(
            (
                "cat",
                Pipeline([
                    ("imp", SimpleImputer(strategy="constant", fill_value="desconocido")),
                    ("enc", OneHotEncoder(handle_unknown="ignore", sparse_output=False, min_frequency=5)),
                ]),
                cols_categorical,
            )
        )

    preprocessor = ColumnTransformer(transformers)
    xgb = XGBRegressor(
        n_estimators=900,
        learning_rate=0.03,
        max_depth=5,
        subsample=0.85,
        colsample_bytree=0.70,
        min_child_weight=5,
        gamma=0.05,
        reg_alpha=0.20,
        reg_lambda=1.4,
        random_state=42,
        n_jobs=-1,
        verbosity=0,
    )
    return Pipeline([
        ("preprocessor", preprocessor),
        (
            "regressor",
            TransformedTargetRegressor(
                regressor=xgb,
                func=np.log1p,
                inverse_func=np.expm1,
            ),
        ),
    ])

def evaluate_predictions(name, y_true, y_pred):
    return {
        "strategy": name,
        "r2": float(r2_score(y_true, y_pred)),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "mape": float(mean_absolute_percentage_error(y_true, y_pred) * 100),
    }

print("\n🚀 Entrenando modelo absoluto...")
t0 = time.time()
absolute_model = build_model()
absolute_model.fit(X_train, y_train, regressor__sample_weight=w_train.values)
y_pred_abs = absolute_model.predict(X_test)
elapsed_abs = time.time() - t0
metrics_abs = evaluate_predictions("absolute", y_test, y_pred_abs)

print("🚀 Entrenando modelo residual calibrado por fuente...")
baseline_train = X_train_enriched["precio_estimado_segmento_area_ajustado"].copy()
baseline_test = X_test_enriched["precio_estimado_segmento_area_ajustado"].copy()
baseline_train = baseline_train.fillna(X_train_enriched["precio_estimado_segmento_area"])
baseline_test = baseline_test.fillna(X_test_enriched["precio_estimado_segmento_area"])
baseline_train = baseline_train.fillna(X_train_enriched["precio_estimado_ciudad_area"])
baseline_test = baseline_test.fillna(X_test_enriched["precio_estimado_ciudad_area"])
baseline_train = baseline_train.fillna(float(y_train.median()))
baseline_test = baseline_test.fillna(float(y_train.median()))
baseline_train = baseline_train.clip(lower=50_000_000)
baseline_test = baseline_test.clip(lower=50_000_000)

residual_train_mask = y_train.notna() & baseline_train.notna() & np.isfinite(baseline_train) & (baseline_train > 0)
print(f"   Residual train válidos: {int(residual_train_mask.sum()):,}/{len(residual_train_mask):,}")
y_train_ratio = (y_train[residual_train_mask] / baseline_train[residual_train_mask]).clip(lower=0.20, upper=4.50)
X_train_res = X_train.loc[residual_train_mask].copy()
w_train_res = w_train.loc[residual_train_mask].copy()

t1 = time.time()
residual_model = build_model()
residual_model.fit(X_train_res, y_train_ratio, regressor__sample_weight=w_train_res.values)
pred_ratio_train = pd.Series(residual_model.predict(X_train_res))
ratio_clip_low = max(0.25, float(pred_ratio_train.quantile(0.01)))
ratio_clip_high = min(4.00, float(pred_ratio_train.quantile(0.99)))
pred_ratio_test = np.clip(residual_model.predict(X_test), ratio_clip_low, ratio_clip_high)
y_pred_res = pred_ratio_test * baseline_test.to_numpy()
elapsed_res = time.time() - t1
metrics_res = evaluate_predictions("residual", y_test, y_pred_res)

comparison = pd.DataFrame([metrics_abs, metrics_res]).sort_values(["mape", "mae"])
print("\n📊 COMPARACIÓN DE ESTRATEGIAS:")
print(comparison.to_string(index=False))

if metrics_res["mape"] < metrics_abs["mape"]:
    best_strategy = "residual"
    best_model = residual_model
    y_pred = y_pred_res
    r2 = metrics_res["r2"]
    mae = metrics_res["mae"]
    mape = metrics_res["mape"]
    elapsed = elapsed_res
else:
    best_strategy = "absolute"
    best_model = absolute_model
    y_pred = y_pred_abs
    r2 = metrics_abs["r2"]
    mae = metrics_abs["mae"]
    mape = metrics_abs["mape"]
    elapsed = elapsed_abs

print(f"\n🏆 Estrategia ganadora: {best_strategy}")
print(f"   R²:   {r2:.4f}")
print(f"   MAE:  ${mae:,.0f} COP")
print(f"   MAPE: {mape:.1f}%")
print(f"   Tiempo: {elapsed:.0f}s")
print("\n⏭️ Cross-validation omitido en esta versión: las features de mercado son fold-dependientes y hacerlo bien requiere recalcularlas dentro de cada fold.")

print("\n📊 MAPE por rango:")
df_eval = pd.DataFrame({"real": y_test.values, "pred": y_pred})
df_eval["rango"] = pd.cut(
    df_eval["real"],
    bins=[0, 300e6, 500e6, 800e6, 1500e6, float("inf")],
    labels=["<300M", "300-500M", "500-800M", "800M-1.5B", ">1.5B"],
)
mape_table = df_eval.groupby("rango", observed=True).apply(
    lambda x: pd.Series({
        "n": len(x),
        "mape_%": round((abs(x.pred - x.real) / x.real * 100).mean(), 1),
        "mae_M": round(abs(x.pred - x.real).mean() / 1e6, 0),
    }),
    include_groups=False,
 )
print(mape_table.to_string())

print("\n" + "=" * 60)
print("  DIAGNÓSTICO DE LEAKAGE")
print("=" * 60)
print("  No se usa precio_num/area_m2 como feature de entrada.")
print("  Las features de mercado se calculan solo desde train y luego se aplican a test.")
print(f"  City stats calculados sobre {len(city_stats):,} ciudades/zonas colapsadas.")
print(f"  Segment stats calculados sobre {len(segment_stats):,} segmentos ciudad+tipo.")
print(f"  Fuente stats calculados sobre {len(fuente_ratio_stats):,} fuentes.")
print(f"  Ciudades no vistas en test resueltas con fallback global: {unseen_test_cities}")
print("  Sample weights: más peso a inmuebles con mayor consenso cross-portal.")
if best_strategy == "residual":
    print(f"  Ratio residual clip: [{ratio_clip_low:.2f}, {ratio_clip_high:.2f}]")
print("=" * 60)

try:
    xgb_step = best_model.named_steps["regressor"].regressor_
    feat_names = best_model.named_steps["preprocessor"].get_feature_names_out()
    imps = pd.Series(xgb_step.feature_importances_, index=feat_names)

    print("\n📊 Top 30 features por importancia:")
    print(imps.sort_values(ascending=False).head(30).to_string())

    groups = {
        "texto_tfidf": imps[imps.index.str.startswith("txt__")].sum(),
        "numericas_total": imps[imps.index.str.startswith("num__")].sum(),
        "categoricas_total": imps[imps.index.str.startswith("cat__")].sum(),
        "num__area_m2": imps[imps.index.str.contains("num__area_m2", regex=False)].sum(),
        "num__log_area_m2": imps[imps.index.str.contains("num__log_area_m2", regex=False)].sum(),
        "num__precio_estimado_segmento_area_ajustado": imps[imps.index.str.contains("num__precio_estimado_segmento_area_ajustado", regex=False)].sum(),
        "num__fuente_segmento_factor": imps[imps.index.str.contains("num__fuente_segmento_factor", regex=False)].sum(),
        "num__num_portales": imps[imps.index.str.contains("num__num_portales", regex=False)].sum(),
        "cat__city_token": imps[imps.index.str.contains("cat__city_token", regex=False)].sum(),
    }

    print("\n📊 Importancia por grupo:")
    for name, imp in sorted(groups.items(), key=lambda item: -item[1]):
        print(f"   {name:40s}: {imp:.4f}  ({imp * 100:.1f}%)")
except Exception as e:
    print(f"No se pudo extraer importancias: {e}")

model_bundle = {
    "model": best_model,
    "strategy": best_strategy,
    "city_stats": city_stats,
    "segment_stats": segment_stats,
    "fuente_ratio_stats": fuente_ratio_stats,
    "fuente_segmento_ratio_stats": fuente_segmento_ratio_stats,
    "market_meta": market_meta,
    "feature_cols": cols_numeric_final + cols_categorical + ["texto_completo"],
    "market_features": market_features,
    "ratio_clip_low": ratio_clip_low if best_strategy == "residual" else None,
    "ratio_clip_high": ratio_clip_high if best_strategy == "residual" else None,
    "trained_at": datetime.now(tz=timezone.utc).isoformat(),
    "metrics": {
        "r2": r2,
        "mae": mae,
        "mape": mape,
        "train_size": len(X_train),
        "test_size": len(X_test),
    },
    "comparison": comparison.to_dict(orient="records"),
    "model_type": "bundle_v3",
}

s3 = boto3.client(
    "s3",
    aws_access_key_id=config["aws_access_key"],
    aws_secret_access_key=config["aws_secret_key"],
)

ts = datetime.now(tz=timezone.utc).strftime("%Y%m%d_%H%M%S")
model_key = f"models/modelo_xgboost_bundle_v3_{ts}.pkl"

buf = BytesIO()
pickle.dump(model_bundle, buf)
buf.seek(0)
s3.upload_fileobj(buf, BUCKET, model_key)
print(f"\n💾 Bundle challenger guardado: s3://{BUCKET}/{model_key}")

manifest_key = "models/manifest.json"
previous_manifest = {}
previous_manifest_exists = False
try:
    previous_manifest = json.loads(s3.get_object(Bucket=BUCKET, Key=manifest_key)["Body"].read())
    previous_manifest_exists = True
    prev_mape = previous_manifest.get("metrics", {}).get("mape", float("inf"))
except Exception:
    prev_mape = float("inf")

should_promote = FORCE_PROMOTE_TO_CHAMPION or (AUTO_PROMOTE_IF_BETTER and mape < prev_mape) or (not previous_manifest_exists)
if should_promote:
    manifest_payload = {
        "champion_model_key": model_key,
        "model_type": model_bundle["model_type"],
        "strategy": best_strategy,
        "trained_at": model_bundle["trained_at"],
        "promoted_at": datetime.now(tz=timezone.utc).isoformat(),
        "metrics": model_bundle["metrics"],
        "comparison": model_bundle["comparison"],
        "promotion_reason": "forced" if FORCE_PROMOTE_TO_CHAMPION else ("better_mape" if previous_manifest_exists else "bootstrap"),
    }
    s3.put_object(
        Bucket=BUCKET,
        Key=manifest_key,
        Body=json.dumps(manifest_payload, indent=2).encode("utf-8"),
        ContentType="application/json",
    )
    print(f"🏆 Manifest actualizado: s3://{BUCKET}/{manifest_key}")
    print(f"   Nuevo principal: {model_key}")
else:
    print(f"📌 Campeón actual: {previous_manifest.get('champion_model_key')} | MAPE {prev_mape:.2f}%")
    print(f"📌 Challenger {best_strategy}/bundle_v3: MAPE {mape:.2f}% | mejora {prev_mape - mape:+.2f} pp")
    print("⚠️ No se promovió a principal porque no mejora al campeón. Usa FORCE_PROMOTE_TO_CHAMPION=True solo si quieres forzarlo.")